In [ ]:
from IPython.display import clear_output
!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()
import os
os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
from datasets import load_dataset
from typing import cast
import torch
import einops
from pathlib import Path

from utils.data import (
    extract_user_instruction,
)

from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed
from mech_interp_toolkit.activation_utils import get_embeddings_dict, get_activations, concat_activations
from safetensors.torch import save_file
import warnings

warnings.filterwarnings("ignore")


set_global_seed(0)
torch.set_grad_enabled(False)

torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [ ]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_train"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "suffix.pt"
batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load 200 examples from the circuit_breakers_train split
dataset = load_dataset(
    dataset_name,
    split=split,
)

dataset = cast(dict, dataset)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]

clear_output()

# Load model, tokenizer and config
model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",  # Scaled Dot Product Attention for efficiency
)

clear_output()


# load suffix
# from rashad's eval code
def load_suffix(suffix_path: str, device: torch.device):
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb


suffix_embed = load_suffix(suffix_path=suffix_path, device=device)
len_suffix = suffix_embed.shape[1]

In [ ]:
num_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(num_layers)]

In [ ]:
# Iterate over prompts_str in batches
num_batches = (len(prompts_str) + batch_size - 1) // batch_size

base_collate = []
new_collate = []

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(prompts_str))

    batch_prompts = prompts_str[start_idx:end_idx]

    current_batch_size = len(batch_prompts)

    print(
        f"Processing batch {batch_idx + 1}/{num_batches} (samples {start_idx} to {end_idx - 1})"
    )

    batch_dict = ch_tokenizer(prompts=batch_prompts)
    # removes "input_ids" and adds "inputs_embeds"
    batch_embeds_dict = get_embeddings_dict(model, batch_dict)
    batch_embeds = batch_embeds_dict["inputs_embeds"]
    batch_attn_mask = batch_embeds_dict["attention_mask"]

    # broadcast suffix
    batch_suffix = einops.repeat(
        suffix_embed,
        "dummy pos d_model -> (curr_batch dummy) pos d_model",
        curr_batch=current_batch_size,
    )

    new_embeds = torch.cat(
        [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
    )
    attn_extension = torch.ones((current_batch_size, len_suffix))
    new_attn = torch.cat([batch_attn_mask, attn_extension], dim=1)

    new_embeds_dict = {
        "inputs_embeds": new_embeds,
        "attention_mask": new_attn
    }

    base_acts = get_activations(
        model, inputs=batch_embeds_dict, layer_components=components, retain_grads=False, positions=None
    ).cpu()

    new_acts = get_activations(
        model, inputs=new_embeds_dict, layer_components=components, retain_grads=False, positions=None
    ).cpu()

    base_collate.append(base_acts)
    new_collate.append(new_acts)

Processing batch 1/47 (samples 0 to 63)
Processing batch 2/47 (samples 64 to 127)
Processing batch 3/47 (samples 128 to 191)
Processing batch 4/47 (samples 192 to 255)
Processing batch 5/47 (samples 256 to 319)
Processing batch 6/47 (samples 320 to 383)
Processing batch 7/47 (samples 384 to 447)
Processing batch 8/47 (samples 448 to 511)
Processing batch 9/47 (samples 512 to 575)
Processing batch 10/47 (samples 576 to 639)
Processing batch 11/47 (samples 640 to 703)
Processing batch 12/47 (samples 704 to 767)
Processing batch 13/47 (samples 768 to 831)
Processing batch 14/47 (samples 832 to 895)
Processing batch 15/47 (samples 896 to 959)
Processing batch 16/47 (samples 960 to 1023)
Processing batch 17/47 (samples 1024 to 1087)
Processing batch 18/47 (samples 1088 to 1151)
Processing batch 19/47 (samples 1152 to 1215)
Processing batch 20/47 (samples 1216 to 1279)
Processing batch 21/47 (samples 1280 to 1343)
Processing batch 22/47 (samples 1344 to 1407)
Processing batch 23/47 (samples 

In [ ]:
# full_base_acts = dict(concat_activations(base_collate, pad_value=0))
# full_new_acts = dict(concat_activations(new_collate, pad_value=0))

In [ ]:
from tqdm import tqdm
save_path = Path(f"outputs/cached_activations/{split}_all_pos.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)

for i, z in enumerate(tqdm(zip(base_collate, new_collate))):
    torch.save(z, save_path.parent / (f"{i}_" + str(save_path.name)))


0it [00:00, ?it/s]
1it [00:02,  2.73s/it]
2it [00:05,  2.91s/it]
3it [00:13,  4.97s/it]
4it [00:18,  5.05s/it]
5it [00:23,  5.17s/it]
6it [00:32,  6.30s/it]
7it [00:42,  7.47s/it]
8it [00:51,  7.97s/it]
9it [01:03,  9.35s/it]
10it [01:14,  9.80s/it]
11it [01:18,  8.03s/it]
12it [01:28,  8.80s/it]
13it [01:33,  7.61s/it]
14it [01:38,  6.79s/it]
15it [01:45,  6.76s/it]
16it [01:50,  6.12s/it]
17it [01:57,  6.55s/it]
18it [02:01,  5.91s/it]
19it [02:08,  6.10s/it]
20it [02:13,  5.83s/it]
21it [02:21,  6.40s/it]
22it [02:27,  6.23s/it]
23it [02:32,  5.90s/it]
24it [02:37,  5.72s/it]
25it [02:43,  5.65s/it]
26it [02:50,  6.00s/it]
27it [02:55,  5.81s/it]
28it [03:03,  6.59s/it]
29it [03:09,  6.22s/it]
30it [03:14,  5.87s/it]
31it [03:20,  6.09s/it]
32it [03:27,  6.30s/it]
33it [03:35,  6.63s/it]
34it [03:41,  6.70s/it]
35it [03:46,  6.21s/it]
36it [03:51,  5.76s/it]
37it [03:58,  5.97s/it]
38it [04:03,  5.88s/it]
39it [04:13,  7.18s/it]
40it [04:20,  7.00s/it]
41it [04:28,  7.26s/it]
42it 

In [ ]:
import shutil
import os

source_dir = 'outputs/cached_activations'
target_dir = '/content/drive/MyDrive'

file_names = os.listdir(source_dir)

for file_name in tqdm(file_names):
    shutil.move(os.path.join(source_dir, file_name), target_dir)
    Path(os.path.join(source_dir, file_name)).unlink(missing_ok=True)


100%|██████████| 47/47 [09:12<00:00, 11.75s/it]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
